# 6.2 — チャンクを越えて正しく集計する


グループの合計と件数を小さな辞書へ保持し、チャンク境界に依存しない集計を作ります。


In [ ]:
from pathlib import Path
import pandas as pd

project = Path.cwd() / "projects" / "clinic-stock-scaleup"
fixture = project / "data" / "clinic-stock-fixture.csv"
print("Project found:", project.is_dir())
print("Fixture found:", fixture.is_file())


In [ ]:
def aggregate(path, chunksize):
    totals = {}
    processed = 0
    for chunk in pd.read_csv(path, chunksize=chunksize):
        processed += len(chunk)
        chunk["stockout_day"] = chunk["stockout_hours"].gt(0).astype(int)
        part = chunk.groupby(["district", "medicine"], as_index=False).agg(
            clinic_days=("date", "size"),
            stockout_days=("stockout_day", "sum"),
            patients_turned_away=("patients_turned_away", "sum"),
        )
        for row in part.itertuples(index=False):
            key = (row.district, row.medicine)
            current = totals.setdefault(key, {"clinic_days": 0, "stockout_days": 0, "patients_turned_away": 0})
            current["clinic_days"] += row.clinic_days
            current["stockout_days"] += row.stockout_days
            current["patients_turned_away"] += row.patients_turned_away
    return processed, totals


## 6.2.1 異なるチャンクサイズを比較する


In [ ]:
processed7, totals7 = aggregate(fixture, 7)
processed13, totals13 = aggregate(fixture, 13)
print("Rows:", processed7, processed13)
print("Same totals:", totals7 == totals13)
assert totals7 == totals13


## 6.2.2 分子と分母から率を作る


In [ ]:
rows = []
for (district, medicine), values in totals7.items():
    rows.append({
        "district": district,
        "medicine": medicine,
        **values,
        "stockout_rate": values["stockout_days"] / values["clinic_days"] * 100,
    })
summary = pd.DataFrame(rows).sort_values("patients_turned_away", ascending=False)
display(summary)


## 確認
チャンクごとの`stockout_rate`を単純平均してはいけない理由を、件数の違いを使って説明してください。
